# TEMREN Görev 3: Görüntü Eşleme - Hiperparametre Optimizasyonu

Bu notebook, Temren takımının Teknofest 2026 Havacılıkta Yapay Zeka Görev 3 (Görüntü Eşleme) algoritmasını (LightGlue + MobileSAM) gerçek drone görüntüleri üzerinde test etmek ve hiperparametrelerini (RANSAC eşiği, güven skoru vb.) optimize etmek için tasarlanmıştır.

**Çalışma Ortamı:** Menüden `Runtime (Çalışma Zamanı)` -> `Change runtime type (Çalışma zamanı türünü değiştir)` seçip donanım hızlandırıcısını **T4 GPU** olarak ayarlayın.

In [ ]:
# 1. GPU Kontrolü
!nvidia-smi

### 2. Repo Klonlama ve Bağımlılıkların Kurulumu

In [ ]:
import os

if not os.path.exists('Temren'):
    !git clone https://github.com/necmettinsnmz/Temren.git

%cd Temren

# En son değişiklikleri çek
!git pull origin main

# Bağımlılıkları Kur
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -r requirements.txt
!pip install git+https://github.com/ChaoningZhang/MobileSAM.git
!pip install -e ./LightGlue

### 3. Drone Test Görüntülerinin İndirilmesi
Burada referans nesne ve o nesnenin farklı açıdan/uzaklıktan çekilmiş hali indirilecektir.

In [ ]:
import urllib.request
import os
import cv2
import matplotlib.pyplot as plt

os.makedirs("test_data", exist_ok=True)

# Test için zorlu açı ve yükseklik farkı içeren örnek görseller
ref_url = "https://raw.githubusercontent.com/cvlab-epfl/lightglue/main/assets/sacre_coeur1.jpg"
frame_url = "https://raw.githubusercontent.com/cvlab-epfl/lightglue/main/assets/sacre_coeur2.jpg"

urllib.request.urlretrieve(ref_url, "test_data/ref.jpg")
urllib.request.urlretrieve(frame_url, "test_data/frame.jpg")

print("Test verileri indirildi!")

### 4. Optimizasyon Öncesi Pipeline Testi

In [ ]:
import sys
sys.path.append('.')

from tasks.task3_matching import ImageMatcher
import time

# Matcher'ı başlat (Model ağırlıkları indirilecektir)
matcher = ImageMatcher(device="cuda")

ref_img = cv2.imread("test_data/ref.jpg")
frame = cv2.imread("test_data/frame.jpg")

# Drone senaryosunu simüle etmek için referans görüntüden bir obje kırpalım
h, w = ref_img.shape[:2]
ref_cropped = ref_img[h//3:2*h//3, w//3:2*w//3].copy()

start_time = time.time()
bbox = matcher.match(frame, ref_cropped)
end_time = time.time()

output = frame.copy()
if bbox is not None:
    x1, y1, x2, y2 = map(int, bbox)
    cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 4)
    
    plt.figure(figsize=(15, 8))
    plt.subplot(1, 2, 1)
    plt.title("Referans Obje (Drone'dan Seçilen)")
    plt.imshow(cv2.cvtColor(ref_cropped, cv2.COLOR_BGR2RGB))
    
    plt.subplot(1, 2, 2)
    plt.title(f"Tespit Süresi: {(end_time - start_time)*1000:.1f}ms")
    plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
    plt.show()
else:
    print("Eşleşme bulunamadı! Lütfen config.py içerisindeki LIGHTGLUE_MIN_MATCHES değerini düşürün.")

### 5. Hyperparameter Tuning (Parametre Optimizasyonu)

Modelin eşleşme performansı düşük gelirse aşağıdaki ayarları değiştirip tekrar deneyin. 
1. Sol menüden `config.py` dosyasını açın.
2. `LIGHTGLUE_MAX_KEYPOINTS` değerini 2048'den **4096**'ya çıkarın (daha detaylı arar, GPU'yu yorar).
3. `LIGHTGLUE_MIN_MATCHES` değerini 15'ten **10**'a düşürün (daha az eşleşmeyle ikna olur).
4. Dosyayı kaydedip 4. Hücreyi tekrar çalıştırın.